# semantic_search/03 — Clinical associations of embedding PCs

Tests retained PCs against demographics, cancer type, stage, metastatic burden, treatment, somatic alterations, PRS, note-volume covariates, and overall survival. Continuous variables use Spearman correlation; categorical variables use Kruskal–Wallis with epsilon-squared; survival uses a univariate Cox model per standardized PC.

BH-FDR is applied across all PC–variable tests within each family. All-time associations are retrospective, including survival associations.

In [ ]:
from __future__ import annotations

import os, subprocess, sys, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import display

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.py').is_file() and (candidate / 'semantic_search').is_dir():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import config
from semantic_search import common

def run_module(module, args):
    cmd = [sys.executable, '-m', module, *args]
    print('$ ' + ' '.join(cmd) + '\n', flush=True)
    started = time.time()
    result = subprocess.run(cmd, cwd=REPO_ROOT)
    print(f'\nexit={result.returncode}  elapsed={(time.time() - started) / 60:,.1f} min')
    return result.returncode

SPACES = common.PC_SPACES
WINDOWS = common.DEFAULT_WINDOWS
MAX_PCS = None  # None tests every retained PC
AVPC_NEPC_LABELS = config.AVPC_NEPC_LABELS_PATH
RUN_TESTS = True

print(f'repo root: {REPO_ROOT}')
print(f'spaces:    {SPACES}')
print(f'windows:   {WINDOWS}')
print(f'PC limit:  {MAX_PCS or "all retained"}')
print(f'LLM labels: {AVPC_NEPC_LABELS}')

## Pre-flight and run

In [ ]:
missing = []
for window in WINDOWS:
    for space in SPACES:
        path = common.pc_scores_path(space, window)
        exists = os.path.exists(path)
        print(f"[{'ok ' if exists else 'MISSING'}] {space}/{window}: {path}")
        if not exists:
            missing.append(path)

if RUN_TESTS and not missing:
    args = ['--spaces', *SPACES, '--windows', *WINDOWS, '--avpc-nepc-labels', AVPC_NEPC_LABELS]
    if MAX_PCS is not None:
        args.extend(['--max-pcs', str(MAX_PCS)])
    return_code = run_module('semantic_search.correlate_pcs', args)
    if return_code:
        raise RuntimeError(f'PC-correlation stage exited with {return_code}')
elif missing:
    print('Run 02_pcs first.')
else:
    print('RUN_TESTS=False; skipped.')

## Coverage and significant associations

In [ ]:
coverage_path = common.result_path('pc_join_coverage')
association_path = common.result_path('pc_clinical_associations')

if os.path.exists(coverage_path):
    coverage = pl.read_csv(coverage_path).filter(pl.col('space').is_in(SPACES) & pl.col('window').is_in(WINDOWS))
    display(coverage.sort(['space', 'window', 'coverage']))

if os.path.exists(association_path):
    associations = pl.read_csv(association_path).filter(pl.col('space').is_in(SPACES) & pl.col('window').is_in(WINDOWS))
    significant = associations.filter(pl.col('significant') == True).sort('fdr')
    print(f'{significant.height:,} FDR-significant PC–clinical associations')
    display(significant.head(100))
else:
    associations = pl.DataFrame()
    print(f'No result at {association_path}')

## Strongest associations

In [ ]:
if associations.height:
    plot_data = associations.filter(pl.col('fdr').is_not_null()).sort('fdr').head(30).with_columns((-pl.col('fdr').clip(lower_bound=1e-300).log10()).alias('minus_log10_fdr'))
    labels = [f"{row['pc']} · {row['variable']}" for row in plot_data.iter_rows(named=True)]
    families = plot_data.get_column('family').unique(maintain_order=True).to_list()
    palette = dict(zip(families, plt.cm.tab10(np.linspace(0, 1, max(1, len(families))))))
    colors = [palette[value] for value in plot_data.get_column('family')]
    fig, ax = plt.subplots(figsize=(9, max(4, 0.25 * plot_data.height)))
    y = np.arange(plot_data.height)
    ax.barh(y, plot_data['minus_log10_fdr'], color=colors)
    ax.set_yticks(y, labels); ax.invert_yaxis(); ax.set_xlabel('−log10(BH FDR)')
    ax.axvline(-np.log10(0.05), color='black', ls='--', lw=1)
    handles = [plt.Line2D([0], [0], color=palette[value], lw=5, label=value) for value in families]
    ax.legend(handles=handles, title='family', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout(); plt.show()
else:
    print('No association results to plot.')